<h>패키지</h>

In [268]:
import pandas as pd
import numpy as np
import requests
import time

# 시각화 패키지
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 경고창 무시
import warnings
warnings.filterwarnings("ignore")

<h>Riot Account ID 정보 불러오기</h>

In [269]:
# API 키 갱신 확인 후 사용
api_key = "RGAPI-bfa150da-1813-4450-96a4-b637fc047bb8"

# 소환사 이름
gameName = "슈 닷"
tagLine = "77777"
account_url = f"https://asia.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}"
account_r = requests.get(account_url)

account_r

<Response [200]>

<p>랭크 유저들의 summonerId 찾기</p>

In [270]:
# 티어는 실버 이상
tier1 = ["SILVER", "GOLD", "PLATINUM", "EMERALD", "DIAMOND"]
tier2 = ["master", "grandmaster", "challenger"]
division = ["I", "II", "III", "IV"]
page = 1

entries = pd.DataFrame()

for t in tier1:

    for div in division:
        rank_entries_url =f"https://kr.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{t}/{div}?page=1&api_key={api_key}"
        rank_entries_r = requests.get(rank_entries_url)

        temp_entries = pd.json_normalize(rank_entries_r.json())

        entries = pd.concat([entries, temp_entries])

for t in tier2:
    rank_entries_url =f"https://kr.api.riotgames.com/lol/league/v4/{t}leagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}"
    rank_entries_r = requests.get(rank_entries_url)

    temp_entries = pd.json_normalize(rank_entries_r.json()['entries'])
    temp_entries['tier'] = t.upper()

    entries = pd.concat([entries, temp_entries])

entries_col_lst = ["tier", "summonerId", "summonerName"]
entries = entries[entries_col_lst]

entries.to_csv("entries.csv", index=False)

<p>summonerId to PUUID</p>

In [273]:
entries = pd.read_csv("entries.csv")
puuids = np.array([], dtype=str)

tier = "EMERALD"
tier_entries = entries[entries['tier'] == tier]

for summonerId in tier_entries['summonerId']:
    puuid_url = f"https://kr.api.riotgames.com/lol/summoner/v4/summoners/{summonerId}?api_key={api_key}"
    puuid_r = requests.get(puuid_url)

    temp_puuid = puuid_r.json()['puuid']
    puuids = np.append(puuids, temp_puuid)

    # RATE LIMITS 피하기
    time.sleep(1)

tier_entries['puuid'] = puuids

tier_entries.to_csv(f"{tier}_entries.csv", index=False)


<h>Find Specific Ranked Matches by PUUID</h>

In [ ]:
tier_entries = pd.read_csv(f"{tier}_entries.csv")

# 매치 정보
tier_matchids = np.array([], dtype=str)
tier_puuids = np.array([], dtype=str)

count = 20

# 각 아이디 당 20개씩 matchid 불러오기
for puuid in tier_entries['puuid']:
    match_url = f"https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start=0&count={count}&api_key={api_key}"
    match_r = requests.get(match_url)
    
    # 한번에 20의 매치 정보
    temp_matchids = match_r.json()

    # puuid 20개씩 스택
    temp_puuids = np.array([], dtype=str)
    for i in range(count):
        temp_puuids = np.append(temp_puuids, puuid)

    # puuid stack
    tier_puuids = np.concatenate([tier_puuids, temp_puuids])
    
    # gameid stack
    tier_matchids = np.concatenate([tier_matchids, temp_matchids])

    # RATE LIMITS 피하기
    time.sleep(1)

tier_puuids_matchids = pd.DataFrame()
tier_puuids_matchids['matchid'] = tier_matchids
tier_puuids_matchids['puuid'] = tier_puuids

<p>Match Info 불러오기</p>

In [324]:
my_game = pd.DataFrame()

for matchid in tier_puuids_matchids['matchid']:

    game_url = f"https://asia.api.riotgames.com/lol/match/v5/matches/{matchid}?api_key={api_key}"
    game_r = requests.get(game_url)

    # info가 없는 게임은 무시
    if 'info' not in game_r.json():
        continue

    # 랭크 게임만 사용
    if game_r.json()['info']['queueId'] != 420:
        continue

    # 랭크 게임만 사용
    if game_r.json()['info']['gameDuration'] < 300:
        continue

    puuid = tier_puuids_matchids[tier_puuids_matchids['matchid'] == matchid]['puuid'].values[0]

    # 10명의 대전 정보
    temp_game = pd.json_normalize(game_r.json()['info']['participants'])
    
    # 나의 participantid 확인
    summoner_lst = pd.json_normalize(game_r.json()['info']['participants'])
    me = summoner_lst[summoner_lst['puuid'] == puuid]
    
    # 나의 대전 정보
    temp_game = temp_game[temp_game['participantId'] == int(me['participantId'])]
    
    # 추가 정보 (게임 종류, 게임 시간)
    temp_game['queueId'] = game_r.json()['info']['queueId']
    temp_game['gameDuration'] = game_r.json()['info']['gameDuration']

    # 상대 라이너의 챔피언 이름
    temp_enemy_champ = summoner_lst[summoner_lst['individualPosition'] == str(me['individualPosition'].values[0])]
    temp_enemy_champ  = temp_enemy_champ[temp_enemy_champ['teamId'] != int(me['teamId'].values[0])]

    # 예외 처리
    if temp_enemy_champ['championName'].size == 0:
        continue

    temp_game['enemychampName'] = str(temp_enemy_champ['championName'].values[0])

    # 대전 정보 stack
    my_game = pd.concat([my_game, temp_game])
    
    # RATE LIMITS 피하기
    time.sleep(1)

# 시간이 오래걸리므로 csv로 저장
my_game.to_csv(f"{tier}_matchdata.csv", index=False)

<p>데이터 전처리</p>

In [325]:
tier_games = pd.read_csv(f"{tier}_matchdata.csv")

# 특정 컬럼만 사용
col_lst = ["teamId", "gameDuration", "championName", "enemychampName",
            "win", "kills", "deaths", "assists", 
            "individualPosition", "item0", "item1", "item2", "item3", "item4", "item5", "item6",
            "perks.statPerks.defense", "perks.statPerks.flex", "perks.statPerks.offense", 
            "perks.styles"]

tier_games2 = tier_games[col_lst]

# KDA 추가하기
def cal_KDA(df):
    # perfect KDA는 death에 1을 추가
    if df["deaths"] == 0:
        adjust = 1
    else:
        adjust = 0
        
    KDA = (df["kills"] + df["assists"]) / (df["deaths"] + adjust)
    
    return round(KDA, 2)
    
tier_games2["K/D/A"] = tier_games2.apply(lambda x: cal_KDA(x), axis=1)

# 컬럼명 변경
rename_col_lst = {
    "teamId": "side", 
    "individualPosition": "position",
    "kills": "K",
    "deaths":"D",
    "assists":"A",
    "K/D/A": "KDA"
}

tier_games2.rename(columns = rename_col_lst, inplace = True)

# 게임종류, 진영 이름 적어주기, 게임시간 분단위
tier_games2["side"] = tier_games2["side"].apply(lambda x: "Blue" if x == 100 else "Red")
tier_games2["gameDuration"] = tier_games2["gameDuration"] / 60

tier_games2.to_csv(f"DP_{tier}_matchdata.csv", index=False)